# DataMate — chạy trên Google Colab

Notebook này chạy toàn bộ solution: engine tất định (CPU) → agent LLM (GPU) → router, đánh giá trên **tập niêm phong** (chỉ in số tổng hợp) và sinh submission cho public test.

**Chuẩn bị** — đặt 3 file vào `MyDrive/datamate/` trên Google Drive (hoặc upload ở bước 2):
`datamate_solution.zip`, `train.zip`, `public_test.zip`

**Runtime** → *Change runtime type* → GPU. T4 (miễn phí) chạy được với model AWQ 4-bit; L4 / A100 (Colab Pro) nhanh hơn nhiều và gần cấu hình thi đấu hơn.
Không có GPU thì các bước 1–5 (rules) vẫn chạy; bỏ qua từ bước 6.

> Colab chỉ có 1 GPU nên `DATAMATE_TP=1`. Môi trường thi là 2×RTX 5090 (TP=2, bf16), nên thời gian ở đây chỉ để ước lượng quy mô.

## 1. Chọn cấu hình theo GPU

In [ ]:
import os, subprocess, json, time, glob, zipfile, shutil, collections

PY = "/content/venv/bin/python"   # môi trường Python 3.12 riêng (Colab mặc định 3.13, vLLM 0.10.1 không hỗ trợ)
CONFIG = "A"   # "A" = Qwen2.5-Coder-7B (sinh code) + Qwen3-4B (chấm)  — 11.6B | "B" = Qwen3-14B cho cả hai vai trò — 14.8B

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True)
GPU = None
if smi.returncode == 0 and smi.stdout.strip():
    name, mem, cc = [x.strip() for x in smi.stdout.splitlines()[0].split(",")]
    GPU = dict(name=name, mem_mb=int(float(mem)), cc=float(cc))
print("GPU:", GPU)

env = dict(DATAMATE_TP="1", PYTHONHASHSEED="0", VLLM_NO_USAGE_STATS="1", TOKENIZERS_PARALLELISM="false",
           DATA_ROOT="/content/data", CACHE_DIR="/content/cache", DM_THINKING="0",
           DM_DEADLINE_S=str(3 * 3600))          # Colab: dừng LLM sau 3h, vẫn giữ đáp án rules
if GPU is None:
    GEN_REPO = VER_REPO = None
    print("Không có GPU -> chỉ chạy mode rules (bước 1-5).")
elif GPU["cc"] < 8.0:                            # T4: không có bf16 -> AWQ 4-bit + fp16, V0 engine, ngữ cảnh ngắn
    GEN_REPO, VER_REPO = {"A": ("Qwen/Qwen2.5-Coder-7B-Instruct-AWQ", "Qwen/Qwen3-4B-AWQ"),
                          "B": ("Qwen/Qwen3-14B-AWQ", None)}[CONFIG]
    env.update(DM_DTYPE="float16", DM_EAGER="1", VLLM_USE_V1="0", DM_GPU_UTIL="0.88",
               DM_MAX_LEN="12288" if CONFIG == "A" else "8192", DM_K_PY="2", DM_K_SQL="1")
elif GPU["mem_mb"] < 30000:                      # L4 24GB
    GEN_REPO, VER_REPO = {"A": ("Qwen/Qwen2.5-Coder-7B-Instruct", "Qwen/Qwen3-4B"),
                          "B": ("Qwen/Qwen3-14B-AWQ", None)}[CONFIG]
    env.update(DM_GPU_UTIL="0.90", DM_MAX_LEN="16384")
elif GPU["mem_mb"] < 50000:                      # A100 40GB
    GEN_REPO, VER_REPO = {"A": ("Qwen/Qwen2.5-Coder-7B-Instruct", "Qwen/Qwen3-4B"),
                          "B": ("Qwen/Qwen3-14B", None)}[CONFIG]
    env.update(DM_GPU_UTIL="0.92", DM_MAX_LEN="32768" if CONFIG == "A" else "16384")
else:                                            # A100 80GB / H100
    GEN_REPO, VER_REPO = {"A": ("Qwen/Qwen2.5-Coder-7B-Instruct", "Qwen/Qwen3-4B"),
                          "B": ("Qwen/Qwen3-14B", None)}[CONFIG]
    env.update(DM_GPU_UTIL="0.92", DM_MAX_LEN="32768")
if GEN_REPO:
    env.update(DM_GEN_MODEL="/content/models/gen", DM_VER_MODEL="/content/models/ver" if VER_REPO else "/content/models/gen")
os.environ.update(env)
print("GEN:", GEN_REPO, "| VER:", VER_REPO or "(dùng chung GEN)")
print({k: v for k, v in env.items() if k.startswith(("DM_", "VLLM", "DATAMATE"))})

## 2. Lấy file (Google Drive hoặc upload)

In [ ]:
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/datamate"     # chứa datamate_solution.zip, train.zip, public_test.zip

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SRC = DRIVE_DIR
    OUT = os.path.join(DRIVE_DIR, "outputs")      # kết quả ghi thẳng lên Drive -> không mất khi Colab ngắt
else:
    from google.colab import files
    SRC = "/content/upload"; os.makedirs(SRC, exist_ok=True); os.chdir(SRC)
    files.upload()                                # chọn cả 3 file zip
    os.chdir("/content")
    OUT = "/content/outputs"
os.makedirs(OUT, exist_ok=True)
os.environ["DM_WORKER_LOG_DIR"] = f"{OUT}/worker_logs"     # log đầy đủ của vLLM mỗi lần gọi model
for f in ["datamate_solution.zip", "train.zip", "public_test.zip"]:
    p = os.path.join(SRC, f); print(f, "OK" if os.path.exists(p) else "THIẾU  <-- kiểm tra lại")

## 3. Giải nén code và dữ liệu

In [ ]:
shutil.rmtree("/content/dm", ignore_errors=True); shutil.rmtree("/content/datamate_solution", ignore_errors=True)
zipfile.ZipFile(os.path.join(SRC, "datamate_solution.zip")).extractall("/content")
shutil.move("/content/datamate_solution", "/content/dm")
os.makedirs("/content/data", exist_ok=True)
for z in ["train.zip", "public_test.zip"]:
    zipfile.ZipFile(os.path.join(SRC, z)).extractall("/content/data")     # -> /content/data/train, /content/data/public_test
os.chdir("/content/dm")
for s in ["train", "public_test"]:
    n = sum(1 for _ in open(f"/content/data/{s}/questions.jsonl", encoding="utf-8"))
    print(s, n, "câu,", len(os.listdir(f"/content/data/{s}/data")), "shop")

## 4. Cài thư viện (venv Python 3.12)
Colab giờ dùng Python 3.13, vLLM 0.10.1 không cài được trên đó; pandas 3 cũng xung đột với gói có sẵn của Colab. Vì vậy dựng một môi trường riêng Python 3.12 bằng `uv` — đúng phiên bản như image Docker. Mất 3–6 phút.

In [ ]:
!pip -q install -U uv
!uv venv /content/venv --python 3.12 --seed -q
pkgs = "vllm==0.10.1 'transformers>=4.55,<4.56' huggingface_hub" if GPU else ""
!uv pip install -q --python {PY} {pkgs} -r /content/dm/requirements.txt
!{PY} -c "import sys, pandas, python_calamine, rapidfuzz; print('python', sys.version.split()[0], '| pandas', pandas.__version__)"
if GPU:
    !{PY} -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

Nếu `uv venv --python 3.12` báo lỗi (không tải được Python 3.12), dùng phương án dự phòng: cài vLLM mới nhất vào Python của Colab.
Chạy cell dưới, **bỏ comment**. Bản này khác phiên bản Docker, nhưng code tương thích. Trên vLLM ≥ 0.11, biến `VLLM_USE_V1=0` không còn tác dụng.

In [ ]:
# !pip -q install -U vllm -r /content/dm/requirements.txt
# PY = "python3"

## 5. Engine tất định (CPU) — hồi quy dev, tập niêm phong, submission rules
`run_eval.py` chỉ chấm 24 shop dev (tự loại 6 shop niêm phong). `eval_sealed.py` chỉ in số tổng hợp — không in câu hay đáp án.

In [ ]:
!{PY} tests/test_agent_mock.py
!{PY} run_eval.py train 2>&1 | grep -E "^N=|MODEL FAIL"
!{PY} eval_sealed.py 2>&1 | tail -5

In [ ]:
# submission chỉ dùng rules — bản an toàn, nộp được ngay
os.makedirs(f"{OUT}/public_rules", exist_ok=True)
t = time.time()
!{PY} predict.py /content/data/public_test {OUT}/public_rules/submission.jsonl --logs {OUT}/public_rules/logs --mode rules 2>&1 | tail -3
print(f"{time.time()-t:.0f}s")
!{PY} validate_submission.py /content/data/public_test/questions.jsonl {OUT}/public_rules/submission.jsonl

## 6. Tải model (GPU)

In [ ]:
from huggingface_hub import snapshot_download
assert GPU, "Cần runtime GPU"
for repo, d in [(GEN_REPO, "/content/models/gen"), (VER_REPO, "/content/models/ver")]:
    if repo:
        snapshot_download(repo, local_dir=d, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.tiktoken", "merges.txt"])
mdirs = " ".join(d for d in ["/content/models/gen", "/content/models/ver"] if os.path.isdir(d))
!{PY} tools/count_params.py {mdirs}
# Rule 2 (<= 15B) tính trên model gốc. Bản AWQ chỉ dùng để chạy thử trên GPU nhỏ; Docker dùng bản bf16.

## 7. Kiểm tra nhanh vLLM
Log in thẳng ra cell. Nếu lỗi trên T4: thử `os.environ['DM_MAX_LEN']='8192'`; nếu vẫn lỗi thì đổi runtime sang L4/A100.

In [ ]:
job = {"id": "x", "messages": [{"role": "user", "content": "Viết 1 dòng pandas tính tổng cột 'tien' của DataFrame df."}],
       "n": 2, "temperature": 0.7, "seed": 0, "max_tokens": 120}
open("/content/job.jsonl", "w").write(json.dumps(job, ensure_ascii=False) + "\n")
extra = ["--enforce-eager"] if os.environ.get("DM_EAGER") == "1" else []
t = time.time()
r = subprocess.run([PY, "-m", "datamate.agent_worker", "--model", os.environ["DM_GEN_MODEL"], "--tp", "1",
                    "--in", "/content/job.jsonl", "--out", "/content/job_out.jsonl",
                    "--max-model-len", os.environ["DM_MAX_LEN"], "--gpu-util", os.environ["DM_GPU_UTIL"],
                    "--dtype", os.environ.get("DM_DTYPE", "auto")] + extra, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-3000:] if r.returncode else "")
print(f"exit={r.returncode}  {time.time()-t:.0f}s (gồm thời gian nạp model)")
if os.path.exists("/content/job_out.jsonl"):
    for txt in json.loads(open("/content/job_out.jsonl").read())["texts"]:
        print("---\n", txt)

## 7b. Chạy thử agent thật trên 12 câu niêm phong
Mất khoảng 3–8 phút. Cần thấy các dòng `[llm] … jobs ok` và `agent answered k/12` với k > 0. Nếu có `WORKER FAILED`, chụp phần log in bên dưới gửi lại.
Hệ thống tự thử lại 3 cấu hình: mặc định → tắt prefix cache → đổi engine V0/V1 + eager.

In [ ]:
!{PY} eval_sealed.py --mode agent --limit 12 > {OUT}/eval_agent12.log 2>&1
!grep -E "^\[llm\]|PA=|^time" {OUT}/eval_agent12.log
!grep -A45 "WORKER FAILED" {OUT}/eval_agent12.log | head -60

## 8. Đánh giá agent + hybrid trên tập niêm phong và chọn ngưỡng

Chạy trên một tập con cố định (`--limit`, chia đều theo shop, seed cố định). `--dump` lưu đầu ra rules/agent (không có đáp án) để `tools/sweep.py` thử các ngưỡng mà không phải gọi lại LLM. Mọi thứ in ra đều là số tổng hợp.

Thời gian ước tính cho 150 câu: T4 khoảng 20–40 phút, A100 khoảng 5–10 phút.

In [ ]:
LIMIT = 150          # tăng lên 300-631 nếu GPU mạnh / còn thời gian (ít nhiễu hơn khi chọn ngưỡng)
DUMP = f"{OUT}/dump_hybrid_{CONFIG}.jsonl"
if os.path.exists(DUMP): os.remove(DUMP)
t = time.time()
!{PY} eval_sealed.py --mode hybrid --limit {LIMIT} --dump {DUMP} > {OUT}/eval_hybrid_{CONFIG}.log 2>&1
!grep -E "^\[llm\]|PA=|unanswerable emitted|by gold|^time" {OUT}/eval_hybrid_{CONFIG}.log
!grep -A45 "WORKER FAILED" {OUT}/eval_hybrid_{CONFIG}.log | head -60
print(f"{(time.time()-t)/60:.1f} phút")
!{PY} tools/sweep.py {DUMP}

**Đọc kết quả sweep**
* Dòng `rules only` là mốc so sánh. Hybrid phải **cao hơn rõ rệt** (≥ 2–3 câu trên 150) thì mới đáng bật; nếu không, giữ submission rules.
* Ít câu nên dễ overfit ngưỡng. Hãy ưu tiên ngưỡng *chặt hơn* trong số các cấu hình gần bằng nhau, và chỉ đổi mặc định khi lợi ích lặp lại trên `LIMIT` lớn hơn.
* Muốn so cấu hình A với B: đổi `CONFIG` ở bước 1, chạy lại các bước 1, 6, 7 và 8 (các bước 2–5 không cần chạy lại), rồi so dòng `BEST`.

In [ ]:
# Đặt ngưỡng đã chọn (để nguyên nếu sweep không cho thấy lợi ích rõ)
os.environ.update(DM_MIN_SHARE="0.6", DM_TAU="0.55", DM_P_TRUST="0.99", DM_TAU_AGENT_ONLY="0.7")

## 9. (Tuỳ chọn) Kiểm thử bền vững: dữ liệu niêm phong bị đổi tên cột / định dạng / nhãn trạng thái / bố cục

In [ ]:
!{PY} tools_perturb.py /content/data headers,values,status,layout /content/variant 2>&1 | tail -2
!{PY} eval_sealed.py --data-variant /content/variant 2>&1 | tail -4
!{PY} eval_sealed.py --data-variant /content/variant --mode hybrid --limit {LIMIT} 2>&1 | tail -4

## 10. Submission public — hybrid
Phase 1 (rules) ghi file ngay; phase 2 (LLM) ghi đè khi xong. Colab ngắt giữa chừng thì file rules vẫn hợp lệ.

In [ ]:
R = f"{OUT}/public_hybrid_{CONFIG}"
os.makedirs(R, exist_ok=True)
t = time.time()
!free -h | head -2
!{PY} predict.py /content/data/public_test {R}/submission.jsonl --logs {R}/logs --mode hybrid > {R}/run.log 2>&1
!grep -E "^\[(phase|done|llm)" {R}/run.log
!grep -A45 "WORKER FAILED\|Traceback" {R}/run.log | head -60
!tail -3 {R}/run.log
print(f"tổng {(time.time()-t)/60:.1f} phút")
!{PY} validate_submission.py /content/data/public_test/questions.jsonl {R}/submission.jsonl

In [ ]:
# Tổng hợp: loại đáp án và kết quả đối chiếu rules <-> agent
S = [json.loads(l) for l in open(f"{R}/submission.jsonl", encoding="utf-8")]
print("loại đáp án:", collections.Counter(s["type"] for s in S).most_common())
L = [json.load(open(f)) for f in glob.glob(f"{R}/logs/*.json")]
print("số file log:", len(L), "(0 nghĩa là predict.py dừng trước khi ghi log — xem run.log)")
print("đối chiếu:", collections.Counter(l.get("cross_check") or ("agent" if l.get("handler") == "agent" else
                                        (l.get("final") or {}).get("type")) for l in L).most_common())
base = {json.loads(l)["id"]: json.loads(l) for l in open(f"{OUT}/public_rules/submission.jsonl", encoding="utf-8")}
print("khác bản rules:", sum(1 for s in S if s != base.get(s["id"])), "câu")

## 11. Biến thể để so trên leaderboard + tải về

`tools/ablation.sh` tạo ba file:
* `sub_A_full` — bản đầy đủ;
* `sub_B_no_unanswerable` — đổi mọi `unanswerable` thành abstain;
* `sub_C_no_llm` — bỏ các câu chỉ agent trả lời.

Nộp A trước. Chỉ nộp B và C khi còn lượt; chênh lệch điểm cho biết phần nào đang có lợi. Đây chỉ là so sánh điểm tổng nên không thu thập đáp án từng câu.

In [ ]:
!bash tools/ablation.sh {R}
shutil.make_archive("/content/datamate_outputs", "zip", OUT)
from google.colab import files
files.download("/content/datamate_outputs.zip")